In [ ]:
import os
import requests
from typing import List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.usage import UsageLimits
from pydantic_ai.models.fallback import FallbackModel

load_dotenv()

class PaperRecommendation(BaseModel):
    paper_title: str
    authors: List[str]
    publication_year: int
    citation_count: int
    citation_source: str
    paper_link: str
    relevance_explanation: str

model = FallbackModel(
    "google:gemini-3.5-flash-lite",
    "google:gemini-3.1-flash-lite",
)

agent = Agent(
    model,
    output_type=PaperRecommendation,
    system_prompt=(
        "You are an expert AI research assistant. Find an academic paper "
        "that strictly matches the user's constraints (topic, publication year, "
        "citation count). You MUST use the search_openalex_papers tool to fetch "
        "real data. NEVER invent or guess citation counts, links, or paper metadata "
        "from memory. Only recommend a paper that was returned by the tool."
    ),
)

def reconstruct_abstract(inverted_index: Optional[dict]) -> str:
    """Converts OpenAlex's abstract_inverted_index into readable text."""
    if not inverted_index:
        return "No abstract available."
    words = []
    for word, positions in inverted_index.items():
        for pos in positions:
            words.append((pos, word))
    words.sort(key=lambda x: x[0])
    return " ".join(word for _, word in words)[:500]  

@agent.tool_plain
def search_openalex_papers(
    query: str,
    min_year: Optional[int] = None,
    max_year: Optional[int] = None,
    exact_year: Optional[int] = None,
    min_citations: Optional[int] = None,
    max_citations: Optional[int] = None,
) -> list[dict]:
    """
    Searches the OpenAlex database for academic papers with optional year and citation filters.
    Returns title, authors, publication_year, citation_count, url/doi, abstract, and source.
    """
    print(
        f"[Tool] Searching OpenAlex: query={query!r}, "
        f"years=({min_year}, {max_year}, {exact_year}), "
        f"citations=({min_citations}, {max_citations})"
    )
    url = "https://api.openalex.org/works"
    
 
    filters = []
    if exact_year is not None:
        filters.append(f"publication_year:{exact_year}")
    else:
        if min_year is not None:
            filters.append(f"publication_year:>{min_year - 1}")
        if max_year is not None:
            filters.append(f"publication_year:<{max_year + 1}")
    if min_citations is not None:
        filters.append(f"cited_by_count:>{min_citations - 1}")
    if max_citations is not None:
        filters.append(f"cited_by_count:<{max_citations + 1}")

    params = {
        "search": query,
        "per-page": 10,
        "sort": "cited_by_count:desc",
    }
    if filters:
        params["filter"] = ",".join(filters)
    if os.getenv("OPENALEX_API_KEY"):
        params["api_key"] = os.getenv("OPENALEX_API_KEY")

    try:
        response = requests.get(url, params=params, timeout=15)
        print(f"[Tool] HTTP status: {response.status_code}")
        if response.status_code == 200:
            results = response.json().get("results", [])
            print(f"[Tool] Returned {len(results)} papers")
            formatted = []
            for p in results:
                authors = [
                    a["author"]["display_name"]
                    for a in p.get("authorships", [])
                    if a.get("author") and a["author"].get("display_name")
                ]
                formatted.append({
                    "title": p.get("title"),
                    "authors": authors[:5],  # Limit author list for readability
                    "publication_year": p.get("publication_year"),
                    "citation_count": p.get("cited_by_count"),
                    "citation_source": "OpenAlex",
                    "url_or_doi": p.get("doi") or p.get("id"),
                    "abstract": reconstruct_abstract(p.get("abstract_inverted_index")),
                })
            return formatted if formatted else [{"message": "No papers matched these exact filters. Try broadening the query."}]
        return [{"error": f"HTTP {response.status_code}: {response.text[:200]}"}]
    except requests.RequestException as exc:
        print(f"[Tool] Request failed: {exc}")
        return [{"error": f"Request failed: {exc}"}]

user_prompt = (
    "Find a research paper about LLM agents for software engineering that was "
    "published after 2022 and has at least 100 citations. Explain why the paper "
    "is relevant and provide the source of the citation count."
)

result = await agent.run(
    user_prompt,
    usage_limits=UsageLimits(request_limit=8),
)

paper = result.output
print(f"\nTitle: {paper.paper_title}")
print(f"Authors: {', '.join(paper.authors)}")
print(f"Year: {paper.publication_year}")
print(f"Citations: {paper.citation_count} (Source: {paper.citation_source})")
print(f"Link: {paper.paper_link}")
print(f"Explanation: {paper.relevance_explanation}")